# Sistem Prediksi Risiko Penyakit Ginjal Kronis Menggunakan Perbandingan Algoritma Machine Learning Berbasis Web

Notebook ini menjadi dokumen eksperimen untuk tugas besar Sistem Cerdas. Fokus penelitian adalah membangun sistem screening edukatif risiko Penyakit Ginjal Kronis atau Chronic Kidney Disease (CKD) menggunakan dataset UCI CKD, membandingkan beberapa algoritma machine learning, lalu mengintegrasikan model terbaik ke aplikasi web.

Catatan: sistem ini bukan alat diagnosis medis. Semua hasil hanya menjelaskan perilaku model pada data pembelajaran.


## 1. Pendahuluan

Penyakit Ginjal Kronis merupakan kondisi serius yang dapat berkaitan dengan beberapa parameter klinis seperti tekanan darah, albumin, serum creatinine, hemoglobin, hipertensi, diabetes mellitus, dan anemia. Dengan dataset yang berisi kombinasi fitur numerik dan kategorikal, pendekatan machine learning dapat digunakan untuk mempelajari pola risiko CKD sebagai demo screening edukatif.

Topik penelitian: **Sistem Prediksi Risiko Penyakit Ginjal Kronis Menggunakan Perbandingan Algoritma Machine Learning Berbasis Web**.


## 2. Rumusan Masalah

1. Bagaimana melakukan preprocessing dataset CKD yang memiliki nilai hilang, fitur numerik, dan fitur kategorikal?
2. Algoritma machine learning mana yang memberikan performa terbaik untuk screening risiko CKD?
3. Bagaimana memastikan preprocessing saat training sama dengan preprocessing saat prediksi di web?
4. Bagaimana menyajikan hasil prediksi model melalui API dan antarmuka web?


## 3. Tujuan Penelitian

1. Menggunakan dataset CKD dari UCI Machine Learning Repository.
2. Membandingkan Logistic Regression, Decision Tree, Random Forest, dan SVC.
3. Memilih model terbaik berdasarkan mean cross-validated F1-score untuk label `ckd`.
4. Menyimpan preprocessing dan model dalam satu scikit-learn `Pipeline`.
5. Menyediakan API dan web UI untuk simulasi screening risiko CKD.


## 4. Dataset dan Sumber Data

Dataset yang digunakan adalah Chronic Kidney Disease Dataset dari UCI Machine Learning Repository.

Sitasi dataset:

Rubini, L., Soundarapandian, P., & Eswaran, P. (2015). Chronic Kidney Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5G020

Dataset memiliki 24 fitur input dan satu target klasifikasi `classification` dengan label `ckd` dan `notckd`.


## 5. Import Library dan Modul Project

Notebook ini tidak membuat pipeline terpisah. Semua proses training dan evaluasi memanggil modul yang sama dengan aplikasi web agar hasil notebook konsisten dengan sistem yang dijalankan.


In [18]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.schemas import SAMPLE_PAYLOAD
from scripts.fetch_data import DATASET_CITATION, DATA_PATH, ensure_dataset
from src.preprocess import (
    CATEGORICAL_FEATURES,
    FEATURE_FIELDS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_pipeline,
)
from src.train import candidate_models, load_dataset, train_and_save

pd.set_option("display.max_columns", 40)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## 6. Import Data

Dataset utama yang dipakai project berada di `data/raw/ckd.csv`. Jika file belum tersedia, fungsi `ensure_dataset()` akan mengambil dan membuat cache dataset.


In [19]:
ensure_dataset()
X, y = load_dataset(DATA_PATH)
df = X.copy()
df[TARGET_COLUMN] = y

print(f"Lokasi dataset: {DATA_PATH}")
print(f"Ukuran fitur X: {X.shape}")
print(f"Ukuran target y: {y.shape}")
display(df.head())


Lokasi dataset: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\data\raw\ckd.csv
Ukuran fitur X: (400, 24)
Ukuran target y: (400,)


,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,blood_urea,serum_creatinine,sodium,potassium,hemoglobin,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,pedal_edema,anemia,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


## 7. Data Understanding

Tahap ini melihat struktur data, tipe fitur, distribusi target, dan jumlah nilai hilang. Informasi ini menentukan strategi preprocessing yang digunakan dalam pipeline.


In [20]:
overview = pd.DataFrame({
    "jumlah_baris": [len(df)],
    "jumlah_fitur": [len(FEATURE_FIELDS)],
    "jumlah_fitur_numerik": [len(NUMERIC_FEATURES)],
    "jumlah_fitur_kategorikal": [len(CATEGORICAL_FEATURES)],
    "target": [TARGET_COLUMN],
})
display(overview)

print("Label target:")
display(y.value_counts().rename_axis("label").reset_index(name="jumlah"))


,jumlah_baris,jumlah_fitur,jumlah_fitur_numerik,jumlah_fitur_kategorikal,target
0,400,24,14,10,classification


Label target:


,label,jumlah
0,ckd,250
1,notckd,150


In [21]:
feature_summary = pd.DataFrame({
    "feature": FEATURE_FIELDS,
    "type": ["numeric" if feature in NUMERIC_FEATURES else "categorical" for feature in FEATURE_FIELDS],
    "missing_count": [int(X[feature].isna().sum()) for feature in FEATURE_FIELDS],
    "missing_percent": [round(float(X[feature].isna().mean() * 100), 2) for feature in FEATURE_FIELDS],
    "unique_count": [int(X[feature].nunique(dropna=True)) for feature in FEATURE_FIELDS],
})
display(feature_summary)


,feature,type,missing_count,missing_percent,unique_count
0,age,numeric,9,2.25,76
1,blood_pressure,numeric,12,3.00,10
2,specific_gravity,numeric,47,11.75,5
3,albumin,numeric,46,11.50,6
4,sugar,numeric,49,12.25,6
5,red_blood_cells,categorical,152,38.00,2
6,pus_cell,categorical,65,16.25,2
7,pus_cell_clumps,categorical,4,1.00,2
8,bacteria,categorical,4,1.00,2
9,blood_glucose_random,numeric,44,11.00,146


In [22]:
print("Statistik fitur numerik:")
display(X[NUMERIC_FEATURES].describe().T)

print("Contoh nilai fitur kategorikal:")
categorical_values = {
    feature: sorted([str(value) for value in X[feature].dropna().unique()])
    for feature in CATEGORICAL_FEATURES
}
display(pd.DataFrame([
    {"feature": feature, "values": ", ".join(values)}
    for feature, values in categorical_values.items()
]))


Statistik fitur numerik:


,count,mean,std,min,25%,50%,75%,max
age,391.0,51.483376,17.169714,2.000,42.00,55.00,64.50,90.000
blood_pressure,388.0,76.469072,13.683637,50.000,70.00,80.00,80.00,180.000
specific_gravity,353.0,1.017408,0.005717,1.005,1.01,1.02,1.02,1.025
albumin,354.0,1.016949,1.352679,0.000,0.00,0.00,2.00,5.000
sugar,351.0,0.450142,1.099191,0.000,0.00,0.00,0.00,5.000
blood_glucose_random,356.0,148.036517,79.281714,22.000,99.00,121.00,163.00,490.000
blood_urea,381.0,57.425722,50.503006,1.500,27.00,42.00,66.00,391.000
serum_creatinine,383.0,3.072454,5.741126,0.400,0.90,1.30,2.80,76.000
sodium,313.0,137.528754,10.408752,4.500,135.00,138.00,142.00,163.000
potassium,312.0,4.627244,3.193904,2.500,3.80,4.40,4.90,47.000


Contoh nilai fitur kategorikal:


,feature,values
0,red_blood_cells,"abnormal, normal"
1,pus_cell,"abnormal, normal"
2,pus_cell_clumps,"notpresent, present"
3,bacteria,"notpresent, present"
4,hypertension,"no, yes"
5,diabetes_mellitus,"no, yes"
6,coronary_artery_disease,"no, yes"
7,appetite,"good, poor"
8,pedal_edema,"no, yes"
9,anemia,"no, yes"


## 8. Visualisasi Dataset dan Distribusi Data

Bagian ini menampilkan visualisasi awal dataset untuk mendukung presentasi akademik: distribusi label target, histogram fitur numerik utama, dan jumlah missing value per fitur. Visualisasi ini membantu menjelaskan karakteristik dataset sebelum preprocessing dan modeling dilakukan.


### 8.1 Distribusi Class Target

Diagram berikut menunjukkan jumlah instance untuk label `ckd` dan `notckd`. Informasi ini penting untuk melihat keseimbangan kelas sebelum model dilatih.


In [ ]:
class_counts = y.value_counts().reindex(["ckd", "notckd"]).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.index, class_counts.values, color=["#2563eb", "#16a34a"])
ax.bar_label(bars, labels=[str(value) for value in class_counts.values], padding=3)
ax.set_title("Distribusi Kelas Target CKD")
ax.set_xlabel("Kelas target")
ax.set_ylabel("Jumlah data")
ax.set_ylim(0, max(class_counts.values) * 1.15)
plt.show()


### 8.2 Histogram Fitur Numerik Penting

Histogram berikut menampilkan sebaran enam fitur numerik yang sering relevan pada analisis CKD: `age`, `blood_pressure`, `blood_glucose_random`, `blood_urea`, `serum_creatinine`, dan `hemoglobin`.


In [ ]:
histogram_features = [
    "age",
    "blood_pressure",
    "blood_glucose_random",
    "blood_urea",
    "serum_creatinine",
    "hemoglobin",
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for ax, feature in zip(axes.ravel(), histogram_features):
    values = pd.to_numeric(df[feature], errors="coerce").dropna()
    ax.hist(values, bins=20, color="#2563eb", edgecolor="white")
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Jumlah data")

fig.suptitle("Histogram Fitur Numerik Penting", fontsize=14)
plt.show()


### 8.3 Missing Values per Fitur

Diagram berikut mengurutkan fitur berdasarkan jumlah missing value terbanyak. Informasi ini menjelaskan alasan penggunaan imputasi pada pipeline preprocessing.


In [ ]:
missing_counts = X[FEATURE_FIELDS].isna().sum().sort_values(ascending=False).head(15)
missing_labels = [feature.replace("_", " ").title() for feature in missing_counts.index]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(missing_labels, missing_counts.values, color="#dc2626")
ax.bar_label(bars, labels=[str(int(value)) for value in missing_counts.values], padding=3)
ax.set_title("Jumlah Missing Value per Fitur Teratas")
ax.set_xlabel("Fitur")
ax.set_ylabel("Jumlah missing value")
ax.tick_params(axis="x", rotation=45)
plt.show()


## 9. Data Preparation

Preprocessing dilakukan di dalam scikit-learn `Pipeline` agar proses training dan prediksi web memakai transformasi yang sama. Strategi preprocessing:

1. Fitur numerik: imputasi median dan standard scaling.
2. Fitur kategorikal: imputasi modus dan one-hot encoding.
3. Model menerima hasil transformasi dari pipeline yang sama.

Dengan pendekatan ini, artifact `pipeline.joblib` berisi preprocessing dan model sekaligus.


In [23]:
models = candidate_models(random_state=42)
model_table = pd.DataFrame([
    {"model_key": name, "estimator": estimator.__class__.__name__}
    for name, estimator in models.items()
])
display(model_table)

example_pipeline = build_pipeline(next(iter(models.values())))
print(example_pipeline)


,model_key,estimator
0,logistic_regression,LogisticRegression
1,decision_tree,DecisionTreeClassifier
2,random_forest,RandomForestClassifier
3,svc_rbf,SVC


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'blood_pressure',
                                                   'specific_gravity',
                                                   'albumin', 'sugar',
                                                   'blood_glucose_random',
                                                   'blood_urea',
                                                   'serum_creatinine', 'sodium',
                                                   'potassium', 'hemoglobin',
                                              

## 10. Modeling dan Perbandingan Algoritma

Fungsi `train_and_save()` menjalankan alur utama:

1. Split data menjadi train dan test set secara stratified.
2. Membandingkan kandidat model menggunakan cross-validation.
3. Memilih model terbaik berdasarkan mean F1-score untuk label `ckd`.
4. Melatih ulang model terbaik pada training set.
5. Mengevaluasi model pada hold-out test set.
6. Menyimpan artifact model, metrik, model card, dan feature importance.


In [24]:
print("Memulai training dan evaluasi model...")
results = train_and_save(random_state=42)
print("Training selesai.")
print(f"Model terbaik: {results['selected_model']}")
print(f"Pipeline tersimpan di: {results['pipeline_path']}")
print(f"Metrik tersimpan di: {results['metrics_path']}")


Memulai training dan evaluasi model...
Training selesai.
Model terbaik: random_forest
Pipeline tersimpan di: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\pipeline.joblib
Metrik tersimpan di: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\metrics.json


## 11. Leaderboard Cross-Validation

Tabel berikut menampilkan performa kandidat model berdasarkan F1-score cross-validation. Model dengan mean F1-score tertinggi dipilih sebagai model akhir.


In [25]:
leaderboard = pd.DataFrame.from_dict(results["metrics"]["cv_scores"], orient="index")
leaderboard = leaderboard.sort_values(by=["mean_f1", "std_f1"], ascending=[False, True])
display(leaderboard[["mean_f1", "std_f1", "scores"]])


,mean_f1,std_f1,scores
random_forest,0.994999,0.006125,"[1.0, 1.0, 0.9873417721518988, 1.0, 0.98765432..."
logistic_regression,0.994872,0.010256,"[1.0, 1.0, 0.9743589743589743, 1.0, 1.0]"
svc_rbf,0.992405,0.006201,"[1.0, 1.0, 0.9873417721518988, 0.9873417721518..."
decision_tree,0.982269,0.017193,"[1.0, 0.9873417721518988, 0.961038961038961, 1..."


## 12. Visualisasi Perbandingan Performa Model

Bar chart berikut membandingkan mean cross-validated F1-score untuk empat algoritma kandidat. Error bar menunjukkan standar deviasi F1-score antar fold cross-validation.


In [ ]:
model_display_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "svc_rbf": "SVC",
}
model_order = [name for name in model_display_names if name in results["metrics"]["cv_scores"]]
mean_f1 = [results["metrics"]["cv_scores"][name]["mean_f1"] for name in model_order]
std_f1 = [results["metrics"]["cv_scores"][name]["std_f1"] for name in model_order]
model_labels = [model_display_names[name] for name in model_order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_labels, mean_f1, yerr=std_f1, capsize=5, color=["#2563eb", "#16a34a", "#dc2626", "#9333ea"])
ax.bar_label(bars, labels=[f"{value:.4f}" for value in mean_f1], padding=3)
ax.set_title("Perbandingan Mean Cross-Validated F1-score")
ax.set_xlabel("Algoritma")
ax.set_ylabel("Mean F1-score untuk label ckd")
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=15)
plt.show()


## 13. Evaluasi Hold-Out Test Set

Setelah model terbaik dipilih melalui cross-validation, model dievaluasi pada test set yang tidak digunakan saat pemilihan model.


In [26]:
metrics = results["metrics"]
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
evaluation_table = pd.DataFrame([
    {"metric": metric_name, "value": metrics.get(metric_name)}
    for metric_name in metric_names
    if metrics.get(metric_name) is not None
])
display(evaluation_table)

print("Label confusion matrix:", metrics.get("labels"))
print("Confusion matrix:")
for row in metrics["confusion_matrix"]:
    print(row)


,metric,value
0,accuracy,0.987500
1,precision,0.980392
2,recall,1.000000
3,f1,0.990099
4,roc_auc,1.000000


Label confusion matrix: ['ckd', 'notckd']
Confusion matrix:
[50, 0]
[1, 29]


## 14. Visualisasi Metrik dan Confusion Matrix Model Terbaik

Bagian ini menampilkan metrik model terbaik dalam bentuk bar chart dan confusion matrix. Visualisasi ini membantu menjelaskan kualitas prediksi model akhir pada hold-out test set.


In [ ]:
metric_display_names = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1-score",
    "roc_auc": "ROC-AUC",
}
metric_order = ["accuracy", "precision", "recall", "f1", "roc_auc"]
metric_values = [np.nan if metrics.get(name) is None else float(metrics[name]) for name in metric_order]
metric_labels = [metric_display_names[name] for name in metric_order]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metric_labels, metric_values, color=["#2563eb", "#16a34a", "#dc2626", "#9333ea", "#ea580c"])
ax.bar_label(
    bars,
    labels=["N/A" if np.isnan(value) else f"{value:.4f}" for value in metric_values],
    padding=3,
)
ax.set_title("Metrik Evaluasi Model Terbaik")
ax.set_xlabel("Metrik")
ax.set_ylabel("Nilai")
ax.set_ylim(0, 1.12)
plt.show()


In [ ]:
confusion = np.array(metrics["confusion_matrix"])
labels = metrics.get("labels", ["ckd", "notckd"])

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion, cmap="Blues")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Confusion Matrix Model Terbaik")
ax.set_xlabel("Prediksi")
ax.set_ylabel("Aktual")
ax.set_xticks(np.arange(len(labels)), labels=labels)
ax.set_yticks(np.arange(len(labels)), labels=labels)

threshold = confusion.max() / 2
for row in range(confusion.shape[0]):
    for col in range(confusion.shape[1]):
        text_color = "white" if confusion[row, col] > threshold else "black"
        ax.text(col, row, str(confusion[row, col]), ha="center", va="center", color=text_color, fontsize=12)

plt.show()


## 15. Feature Importance

Feature importance dihitung dengan permutation feature importance. Nilai ini membantu melihat fitur apa yang paling memengaruhi performa model pada data evaluasi.


In [27]:
feature_importance_path = Path(results["feature_importance_path"])
feature_importance_data = json.loads(feature_importance_path.read_text(encoding="utf-8"))

print(f"Metode: {feature_importance_data.get('method')}")
print(f"Positive label: {feature_importance_data.get('positive_label')}")

top_features = pd.DataFrame(feature_importance_data.get("top_features", []))
display(top_features.head(10))


Metode: permutation_importance
Positive label: ckd


,feature,importance,std
0,specific_gravity,0.027838,0.014377
1,serum_creatinine,0.010602,0.007982
2,hemoglobin,0.009631,0.008663
3,hypertension,0.007728,0.007204
4,diabetes_mellitus,0.007728,0.007204
5,age,0.006795,0.004448
6,sodium,0.006795,0.004448
7,potassium,0.006795,0.004448
8,albumin,0.006757,0.007519
9,appetite,0.003883,0.004755


## 16. Visualisasi Feature Importance

Horizontal bar chart berikut menampilkan 10 fitur teratas berdasarkan permutation feature importance. Nilai ini menjelaskan kontribusi fitur terhadap perilaku model pada data evaluasi, bukan hubungan sebab-akibat medis.


In [ ]:
feature_importance_top10 = top_features.head(10).copy()
feature_importance_top10 = feature_importance_top10.sort_values("importance", ascending=True)
feature_labels = [feature.replace("_", " ").title() for feature in feature_importance_top10["feature"]]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    feature_labels,
    feature_importance_top10["importance"],
    xerr=feature_importance_top10.get("std"),
    capsize=4,
    color="#16a34a",
)
ax.bar_label(bars, labels=[f"{value:.4f}" for value in feature_importance_top10["importance"]], padding=3)
ax.set_title("Top 10 Feature Importance Model Terbaik")
ax.set_xlabel("Permutation importance")
ax.set_ylabel("Fitur")
plt.show()


## 17. Diagram Alur Training Model

Alur training model pada project ini mengikuti pipeline yang sama antara notebook dan kode produksi.

```mermaid
flowchart LR
    A[Dataset UCI CKD] --> B[Preprocessing]
    B --> C[Train/Test Split]
    C --> D[Cross-Validation]
    D --> E[Model Comparison]
    E --> F[Selected Model]
    F --> G[Evaluation]
    G --> H[Save Artifacts]
    H --> I[Prediction API]
```


## 18. Diagram Arsitektur Sistem

Diagram berikut merangkum hubungan antara pengguna, web UI, API, service prediksi, pipeline model, dan output prediksi.

```mermaid
flowchart LR
    A[User] --> B[Web UI]
    B --> C[FastAPI]
    C --> D[Predictor Service]
    D --> E[Pipeline Model]
    E --> F[Output Prediksi]
```


## 19. Ekspor Figure Presentasi dan Jurnal

Cell berikut menjalankan script visualisasi untuk menyimpan semua figure ke folder `reports/figures/`. File PNG tersebut dapat digunakan kembali untuk PPT, laporan, atau draft publikasi.


In [ ]:
completed = subprocess.run(
    [sys.executable, "scripts/generate_figures.py"],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)


## 20. Simulasi Prediksi

Cell berikut menjalankan contoh prediksi menggunakan payload yang sama dengan schema API. Ini memastikan artifact pipeline dapat menerima data berbentuk tabel dengan nama fitur yang sesuai.


In [28]:
pipeline = joblib.load(results["pipeline_path"])
sample_input = pd.DataFrame([SAMPLE_PAYLOAD])
prediction = pipeline.predict(sample_input)[0]

probability = None
if hasattr(pipeline, "predict_proba"):
    probabilities = pipeline.predict_proba(sample_input)[0]
    classes = list(pipeline.classes_)
    probability = float(probabilities[classes.index(prediction)])

print("Contoh input:")
display(sample_input)
print(f"Prediksi model: {prediction}")
if probability is not None:
    print(f"Probabilitas label prediksi: {probability:.4f}")
print("Catatan: hasil ini adalah screening edukatif, bukan diagnosis medis.")


Contoh input:


,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,blood_urea,serum_creatinine,sodium,potassium,hemoglobin,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,pedal_edema,anemia
0,55,80,1.02,0,0,normal,normal,notpresent,notpresent,120,40,1.2,137,4.5,14.5,45,8000,5.2,no,no,no,good,no,no


Prediksi model: notckd
Probabilitas label prediksi: 0.9825
Catatan: hasil ini adalah screening edukatif, bukan diagnosis medis.


## 21. Integrasi Web dan API

Artifact yang dibuat notebook digunakan oleh aplikasi web melalui FastAPI. Jalankan server dari root repository:

```powershell
uvicorn app.main:app --reload --port 8000
```

Endpoint utama:

- `POST /api/v1/screen`
- `GET /api/v1/metrics`
- `GET /api/v1/model-info`
- `GET /api/v1/health`

Setelah server aktif, buka `http://127.0.0.1:8000` untuk mencoba web UI.


## 22. Artifact Penelitian

Cell berikut memeriksa file artifact yang dibutuhkan aplikasi web.


In [29]:
artifact_paths = {
    "pipeline": results["pipeline_path"],
    "metrics": results["metrics_path"],
    "model_card": results["model_card_path"],
    "feature_importance": results["feature_importance_path"],
}

for name, path in artifact_paths.items():
    artifact_path = Path(path)
    print(f"{name}: {artifact_path} | exists={artifact_path.exists()}")


pipeline: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\pipeline.joblib | exists=True
metrics: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\metrics.json | exists=True
model_card: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\model_card.json | exists=True
feature_importance: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\app\artifacts\feature_importance.json | exists=True


## 23. Kesimpulan

Notebook ini menunjukkan bahwa sistem CKD dibangun dengan alur penelitian yang jelas: data understanding, preprocessing, perbandingan algoritma, evaluasi model terbaik, penyimpanan artifact, dan integrasi web. Model terbaik dipilih berdasarkan cross-validated F1-score untuk label `ckd`, lalu disimpan sebagai satu pipeline agar preprocessing dan inferensi tetap konsisten.

Batasan utama:

1. Dataset berukuran kecil sehingga metrik tinggi perlu dibaca hati-hati.
2. Sistem belum melalui validasi klinis.
3. Output adalah screening edukatif, bukan diagnosis medis.
4. Keputusan kesehatan tetap memerlukan pemeriksaan tenaga medis profesional.
